## Assessment solution key

Use this separate notebook after attempting Notebook 9. It supplies the three LangServe endpoints expected by the assessment frontend. It is not applied by the language/mode switcher.

First run Notebook 7 and its saved-index reload cell. Stop any other server using port 9012 before running this key.


In [ ]:
from pathlib import Path
import os

# Jupyter starts this notebook inside the solutions folder.
if Path.cwd().name == "solutions":
    os.chdir("..")
assert Path("docstore_index/index.faiss").is_file(), "Run Notebook 7 and reload its saved index first."


In [ ]:
%%writefile assessment_solution_server.py
from pathlib import Path
from fastapi import FastAPI
from langserve import add_routes
from langchain_nvidia_ai_endpoints import ChatNVIDIA, NVIDIAEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from composer.documents import validate_documents

embedder = NVIDIAEmbeddings(model="course/embedding", base_url="http://llm_client:9000/v1")
llm = ChatNVIDIA(model="nvidia/nemotron-3.5-lightning-30b-a3b",
                 model_kwargs={"chat_template_kwargs": {"enable_thinking": False}})
docstore = FAISS.load_local("docstore_index", embedder, allow_dangerous_deserialization=True)
validate_documents(list(docstore.docstore._dict.values()))

# The frontend performs retrieval and passes the resulting context to the generator.
retriever = docstore.as_retriever(search_kwargs={"k": 4})
prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer using the supplied context. State when it is insufficient. "
               "Preserve exact identifiers and numbers, and cite the source when supplied.\n\n{context}"),
    ("user", "{input}"),
])
generator = prompt | llm | StrOutputParser()

app = FastAPI(title="RAG assessment solution")
add_routes(app, llm, path="/basic_chat")
add_routes(app, retriever, path="/retriever")
add_routes(app, generator, path="/generator")

if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=9012)


In [ ]:
# This cell serves requests until you interrupt it.
!python assessment_solution_server.py


Open the [course frontend](/8090/), select **RAG**, and click **Evaluate**. Read any retrieval or generation failure before retrying. After the checks pass, use **Assess Task** in the course launcher.
